# Task 4 – Teil 3b: Spannungsregelung

**Ausgangspunkt:** das ausgelegte Netz aus `Task4-3_Ortsnetz_Auslegung.ipynb` (CIGRE-MV ohne DER, CIGRE-LV-Ortsnetz an Bus 12 mit pandapower-Standardtrafos und verstärkter Freileitung im Gewerbenetz). Dieses Notebook setzt voraus, dass Teil 3 einmal ausgeführt wurde (Simulationen `w4_t3_ausgelegt` und `w4_t3_000` … `w4_t3_111`).

**Aufbau:**

*Teil A – Regelung des Ortsnetzes mit regelbaren Ortsnetztransformatoren (rONT)*

1. Bewertung möglicher Regelungsmethoden
2. Regelbedarf im Ortsnetz
3. Regelkonzept rONT
4. Kontrollierte Untersuchung ausgewählter Stufenstellungen
5. Zeitreihe ungeregelt und geregelt

*Teil B – Backup-Plan für das MS-Netz*

6. Backup-Plan mit den Trennschaltern S1–S3: Literatur, Wiederversorgung nach Ausfall ((n-1)-Prüfung), Spannungsstützung

*Teil C – Zusammenspiel*

7. Simulation mit rONT-Regelung und Backup-Plan gleichzeitig
8. Fazit

## Importe

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandapower as pp

import simulation as sim
import szenarien as sz
import lvnetz
import regelung as rg
import netplan
import plots
from elements import Diagram
from profiles import Month

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.tab20.colors)

sim.ONLY_RUN_IF_CHANGED = True

V_MIN, V_MAX = 0.90, 1.10                 # Arbeitsgrenzen
V_PLAN_MIN, V_PLAN_MAX = 0.95, 1.05       # planerisches Spannungsband (Teil 3)
PLAN_MARKER = [("orange", V_PLAN_MIN, "Planungsband 0,95 pu", "--"), ("orange", V_PLAN_MAX, "Planungsband 1,05 pu", "--")]

ausgelegt = sim.loadSimulation("w4_t3_ausgelegt")        # aus Task4-3 (ausgelegtes Netz, alle Trennschalter offen)
ungeregelt = {k: sim.loadSimulation(f"w4_t3_{k}") for k in sz.KEYS}   # Zeitreihen aller Schalterstellungen ohne Regelung
g = lvnetz.bus_groups(ausgelegt.net)
TEILNETZE = {x: [b for b in g["NS"] if b.startswith(f"LV Bus {x}")] for x in "RIC"}
TRAFO = {x: f"LV Trafo {x}0-{x}1" for x in "RIC"}

def band_stunden(s):
    """Stunden außerhalb des Planungsbands und der Arbeitsgrenzen, getrennt nach MS und NS."""
    vm = s.results.vm_pu[s.results.converged]
    out = {}
    for lvl, cols in (("MS", g["MS"]), ("NS", g["NS"])):
        lo, hi = vm[cols].min(axis=1), vm[cols].max(axis=1)
        out[f"Stunden < {V_PLAN_MIN} {lvl}"] = int((lo < V_PLAN_MIN).sum())
        out[f"Stunden > {V_PLAN_MAX} {lvl}"] = int((hi > V_PLAN_MAX).sum())
        out[f"Stunden < {V_MIN} {lvl}"] = int((lo < V_MIN).sum())
        out[f"Stunden > {V_MAX} {lvl}"] = int((hi > V_MAX).sum())
        out[f"U min {lvl} [pu]"] = lo.min()
        out[f"U max {lvl} [pu]"] = hi.max()
    return pd.Series(out)

## 1 Bewertung möglicher Regelungsmethoden

Betrachtet wurden drei Ansätze für die Regelung:

| | 1. Schwellwertregelung | 2. Fahrplan aus historischen Werten | 3. Prognose mit neuronalem Netz |
|---|---|---|---|
| Prinzip | reagiert auf den Messwert, wenn eine Grenze überschritten wird | Stufenstellung nach Zeitplan, abgeleitet z. B. aus dem Vorjahr | Stufenstellung nach Vorhersage des Verbrauchs der nächsten Stunde |
| Aufwand in der Simulation | gering bis mittel | gering (Stufen je Monat sind in `runSimulation(tap=...)` schon möglich) | hoch (Trainingsdaten, Merkmale, Validierung) |
| Stand der Technik | ja, so arbeiten Spannungsregler von Stufenschaltern [19, S. 714], [20], [21] | als Ergänzung | Forschung, Smart Grid |
| Schwäche | – (siehe unten) | reagiert nicht auf Abweichungen; nach dem Arbeitsauftrag keine automatische Regelung | nur ein synthetisches Jahr als Datenbasis |

**Zu 1:** Der befürchtete Nachteil, dass mindestens eine Stunde über dem Schwellwert liegt, entsteht nur durch die Stundenauflösung der Simulation. Ein realer Stufenschalter reagiert nach einer Verzögerungszeit im Bereich von Sekunden bis Minuten; der ECOTAP VPD schafft bis zu 20 Stufenschaltungen pro Minute [21]. In der Simulation wird das abgebildet, indem nach einem Eingriff **dieselbe Stunde** erneut gerechnet wird (wie bei den Controllern von pandapower). Damit bleibt keine Stunde verletzt, solange der Stellbereich reicht.

**Zu 2:** Die BDEW-Standardlastprofile sind in jedem Jahr gleich. Ein Fahrplan aus dem „Vorjahr“ wäre deshalb automatisch perfekt und damit nicht aussagekräftig. Außerdem verlangt der Arbeitsauftrag ausdrücklich eine automatische Regelung; eine fest vorgegebene Stufenstellung genügt nicht.

**Zu 3:** Mit nur einem synthetischen Jahr ist eine Verbrauchsprognose nahezu trivial und kaum belastbar. Der Nutzen gegenüber 1 ist gering, weil der Stufenschalter ohnehin innerhalb der Stunde reagiert. Die Idee passt aber in den Ausblick (Teil 9), z. B. als Prognose für Speicher und steuerbare Lasten.

**Entscheidung:** Methode 1 mit Totband, Hysterese, Stellgrenzen und Zählung der Schaltvorgänge. Methoden 2 und 3 werden nicht umgesetzt; 3 wird im Ausblick aufgegriffen.

---
# Teil A – Regelung des Ortsnetzes (rONT)

## 2 Regelbedarf im Ortsnetz

Je Teilnetz: niedrigste Spannung an der NS-Sammelschiene (Trafo-Unterspannungsseite) und am schwächsten Bus, Stunden unter 0,95 pu und die Zahl der Stunden, in denen eine bzw. zwei Stufen (je 2,5 %) nötig wären (Zeitreihe aus Teil 3, ohne Regelung):

In [ ]:
vm0 = ungeregelt["000"].results.vm_pu
bedarf = {}
for x, busse in TEILNETZE.items():
    lo, hi = vm0[busse].min(axis=1), vm0[busse].max(axis=1)
    bedarf[f"Teilnetz {x}"] = {
        "Sammelschiene min [pu]": vm0[f"LV Bus {x}1"].min(),
        "schwächster Bus": vm0[busse].min().idxmin(),
        "schwächster Bus min [pu]": lo.min(),
        "Stunden < 0,95": int((lo < 0.95).sum()),
        "1 Stufe nötig": int(((lo < 0.95) & (lo >= 0.925)).sum()),
        "2 Stufen nötig": int((lo < 0.925).sum()),
        "U max [pu]": hi.max(),
    }
pd.DataFrame(bedarf).T

In [ ]:
vergleich_messpunkt = ["LV Bus R1", "LV Bus R15", "LV Bus C1", "LV Bus C12"]
ungeregelt["000"].createDiagram(Diagram.VOLTAGE, vergleich_messpunkt, months=Month.JANUARY, markers=PLAN_MARKER,
                                title="Sammelschiene (R1, C1) und Strangende (R15, C12) ohne Regelung")

**Folgerungen aus der Analyse:**

- **Messort:** An den NS-Sammelschienen fällt die Spannung nie unter 0,984 pu. Die Verletzungen des Planungsbands treten nur an den Strangenden auf (LV Bus R15 im Wohngebiet, LV Bus C12 im Gewerbe): abends (und am Wochenende mittags) im Wohngebiet, werktags tagsüber im Gewerbe. Ein Regler, der nur die Sammelschiene misst, würde nie eingreifen. Deshalb misst der Regler die niedrigste und höchste Spannung im ganzen Teilnetz.
- **Regelbedarf:** Im Wohngebiet 195 Stunden, im Gewerbe 255 Stunden unter 0,95 pu; fast immer reicht eine Stufe (2,5 %), nur in 2 Stunden wären zwei Stufen nötig. Das Industrienetz liegt immer im Band.

## 3 Regelkonzept rONT

| Merkmal | Festlegung | Begründung |
|---|---|---|
| geregelte Trafos | alle drei Ortsnetztrafos, jeder mit eigenem Regler | jeder Trafo wirkt nur auf sein eigenes Teilnetz; die Teilnetze haben unterschiedliche Lastgänge (Haushalt abends, Gewerbe vormittags) |
| wirksam für | nur die NS-Busse des eigenen Teilnetzes | die US-Spannung eines Trafos ändert die Spannungen im MV-Netz und in den anderen Teilnetzen praktisch nicht |
| Stufenseite | Oberspannung (`tap_side = "hv"`) | Regelwicklung üblicherweise auf der OS-Seite [19, S. 421]; Standardtyp |
| Stellbereich | −2 … +2 Stufen zu je 2,5 % | Standardtyp; entspricht dem rONT-Beispiel ±2 × 2,5 % [20] |
| Messgröße | niedrigste und höchste Spannung aller NS-Busse des Teilnetzes | entspricht Sensoren an den Strangenden; an der Sammelschiene allein wäre keine Verletzung zu sehen (Abschnitt 2) |
| Sollband | 0,95 … 1,05 pu (Planungsband) | Teil 3 |
| Regelgesetz | Abweichung bis 2,5 % unter bzw. über dem Band → 1 Stufe; bis 5 % → 2 Stufen | Stufen entsprechen der Abweichung |
| Stellgrenze | Anheben nur, wenn U_max danach ≤ 1,05 pu bleibt; Absenken nur, wenn U_min ≥ 0,95 pu bleibt | Behebung einer Verletzung darf keine neue erzeugen |
| Rückstellung / Hysterese | eine Stufe zurück Richtung Neutralstellung nur, wenn danach beide Grenzen mit 0,5 % Abstand im Band liegen; Wirkung je Stufe vorsichtig mit 1,2 × 2,5 % geschätzt | verhindert Hin- und Herschalten |
| Verzögerung | in der Simulation: Eingriff innerhalb derselben Stunde (Stunde wird neu gerechnet); real: Verzögerungszeit im Bereich von Sekunden bis Minuten [21] | Stundenauflösung |
| Vorzeichen | `tap_pos = −1` hebt die US-Spannung um eine Stufe an | geprüft mit pandapower |

Der Regler berücksichtigt Unter- und Überspannung. Ohne Erzeuger tritt Überspannung bisher nicht auf; mit den Erzeugungsszenarien in Teil 4 und bei Leistungsflussumkehr wirkt dieselbe Logik nach unten.

**Umsetzung:** `regelung.py` enthält die Klasse `TapRegler`. Die Regler werden an `runSimulation(controllers=[...])` übergeben. Nach jedem Lastfluss prüft jeder Regler das Ergebnis; greift einer ein, wird dieselbe Stunde neu gerechnet, bis keiner mehr eingreift (höchstens 10-mal). Die Stufenstellungen werden in die nächste Stunde übernommen und je Stunde gespeichert (`results.tap_pos`).

In [ ]:
def lv_regler():
    return [rg.TapRegler(TRAFO[x], TEILNETZE[x], u_min=V_PLAN_MIN, u_max=V_PLAN_MAX) for x in "RIC"]

for r in lv_regler():
    print(r.trafo, "→", len(r.busse), "überwachte NS-Busse")

## 4 Kontrollierte Untersuchung ausgewählter Stufenstellungen

Bevor automatisch geregelt wird, wird kontrolliert untersucht, wie feste Stufenstellungen (−2 … +2, an allen drei Ortsnetztrafos gleich) die Spannungen verändern. Zwei statische Betriebspunkte:

- **Spitzenlast:** alle Lasten mit Bemessungsleistung
- **Schwachlast:** alle Lasten mit 25 % der Bemessungsleistung (etwa die Nachtlast)

In [ ]:
def stufen_test(faktor):
    rows = {}
    for tap in range(-2, 3):
        net = copy.deepcopy(ausgelegt.net)
        net.load[["p_mw", "q_mvar"]] *= faktor
        for x in "RIC":
            net.trafo.loc[net.trafo.name == TRAFO[x], "tap_pos"] = tap
        pp.runpp(net)
        vm = pd.Series(net.res_bus.vm_pu.to_numpy(), index=net.bus.name)
        row = {"U min MS [pu]": vm[g["MS"]].min(), "U max MS [pu]": vm[g["MS"]].max()}
        for x, busse in TEILNETZE.items():
            row[f"U min {x} [pu]"] = vm[busse].min()
            row[f"U max {x} [pu]"] = vm[busse].max()
        rows[tap] = row
    out = pd.DataFrame(rows).T
    out.index.name = "Stufe (tap_pos)"
    return out

spitze = stufen_test(1.0)
schwach = stufen_test(0.25)
display(plots.table(spitze, decimals=3, cmap="RdYlGn", vmin=0.90, vmax=1.03))
plots.table(schwach, decimals=3, cmap="RdYlGn_r", vmin=0.97, vmax=1.10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, (titel, df) in zip(axes, (("Spitzenlast", spitze), ("Schwachlast (25 %)", schwach))):
    for x, farbe in zip("RIC", ("tab:blue", "tab:green", "tab:orange")):
        ax.fill_between(df.index, df[f"U min {x} [pu]"], df[f"U max {x} [pu]"], alpha=0.25, color=farbe, label=f"Teilnetz {x}")
        ax.plot(df.index, df[f"U min {x} [pu]"], "o-", color=farbe)
        ax.plot(df.index, df[f"U max {x} [pu]"], "s--", color=farbe)
    ax.plot(df.index, df["U min MS [pu]"], "k^-", label="MS min")
    for v, c, ls in ((V_MIN, "red", "--"), (V_MAX, "red", "--"), (V_PLAN_MIN, "orange", ":"), (V_PLAN_MAX, "orange", ":")):
        ax.axhline(v, color=c, ls=ls, lw=1)
    ax.set_xticks(range(-2, 3))
    ax.set_xlabel("Stufe tap_pos (−: US-Spannung höher)")
    ax.set_title(titel)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Spannung [pu] (Fläche: min … max im Teilnetz)")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

**Ergebnis der kontrollierten Untersuchung:**

- Eine Stufe verändert die NS-Spannungen im ganzen Teilnetz um etwa 2,7 bis 2,8 Prozentpunkte (etwas mehr als die nominellen 2,5 %, weil auch der Spannungsfall im Trafo und in den Leitungen mit der höheren Spannung sinkt). Die MS-Spannungen ändern sich nicht.
- **Spitzenlast:** Ohne Stufe liegen Wohngebiet und Gewerbe bei 0,913 bzw. 0,917 pu. Eine Stufe (−1) reicht mit 0,941 bzw. 0,945 pu noch nicht ganz, zwei Stufen (−2) heben sie auf 0,971 bzw. 0,974 pu.
- **Schwachlast:** Mit zwei Stufen (−2) würden die NS-Spannungen auf bis zu 1,073 pu steigen und das Planungsband überschreiten; mit einer Stufe (−1) bleiben sie bei 1,045 pu im Band.
- **Folgerung:** Keine feste Stufenstellung passt für beide Betriebspunkte. Eine Stellung, die die Spitzenlast abdeckt, erzeugt bei Schwachlast Überspannung. Deshalb ist eine automatische Regelung nötig, die die Stufe mit der Last nachführt.

## 5 Zeitreihe ungeregelt und geregelt

### 5.1 Normalschaltzustand

Gleiche Randbedingungen wie die ungeregelte Zeitreihe `w4_t3_000` (S1–S3 offen, Normalschaltzustand); zusätzlich die drei rONT-Regler.

In [ ]:
geregelt = ausgelegt.copy("w4_t3_geregelt", overwrite=True)
geregelt.runSimulation(controllers=lv_regler())

In [ ]:
vgl = pd.DataFrame({"ungeregelt": band_stunden(ungeregelt["000"]), "geregelt (rONT)": band_stunden(geregelt)})
display(plots.table(vgl.astype(float), decimals=3))
display(rg.schaltvorgaenge(geregelt).filter(like="Stufen").to_frame("Anzahl in 12 Wochen"))
print("Stunden mit Reglereingriff:", int((geregelt.results.info.regler_iterationen > 0).sum()), "von", len(geregelt.results.info))

In [ ]:
taps = geregelt.results.tap_pos[[TRAFO[x] for x in "RIC"]]
haeufigkeit = taps.apply(pd.Series.value_counts).fillna(0).astype(int)
haeufigkeit.index.name = "Stufe (Stunden)"
display(haeufigkeit)

fig, ax = plt.subplots(figsize=(13, 3.2))
idx = geregelt.results.info.index[geregelt.results.info.month == "JANUARY"]
for x, farbe in zip("RIC", ("tab:blue", "tab:green", "tab:orange")):
    ax.step(idx, taps.loc[idx, TRAFO[x]], where="post", color=farbe, label=TRAFO[x])
plots.week_axis(ax, idx)
ax.set_ylabel("Stufe tap_pos")
ax.set_yticks(range(-2, 3))
ax.set_title("Stufenstellungen der Ortsnetztrafos – Januar", pad=22)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
schwaechste = [vm0[TEILNETZE[x]].min().idxmin() for x in "RC"]
sim.compare([ungeregelt["000"], geregelt], Diagram.VOLTAGE, schwaechste, months=Month.JANUARY, markers=PLAN_MARKER,
            title="Schwächste NS-Busse – ungeregelt und geregelt")
geregelt.createDiagram(Diagram.VOLTAGE, g["NS"], layout="boxplot", markers=PLAN_MARKER, title="NS-Spannungen geregelt")

In [ ]:
geregelt.createDiagram(Diagram.NETWORK, time=["min", "max_loading"], title="Geregeltes Netz")

**Ergebnis ungeregelt gegenüber geregelt:**

| Kennwert | ungeregelt | geregelt |
|---|---|---|
| Stunden NS < 0,95 pu | 450 | **0** |
| niedrigste NS-Spannung | 0,923 pu | **0,950 pu** |
| höchste NS-Spannung | 1,021 pu | 1,027 pu |
| Verletzungen der Arbeitsgrenzen | 0 | 0 |
| Stufenschritte in 12 Wochen | – | R: 92, I: 0, C: 100 |

- Die Regler beseitigen alle 450 Stunden unter dem Planungsband im Ortsnetz. Sie greifen in 187 der 2016 Stunden ein und nutzen nur die Stufe −1 (Wohngebiet 238 Stunden, Gewerbe 335 Stunden); den Rest der Zeit steht der Stufenschalter in Neutralstellung. Eine neue Überspannung entsteht nicht (höchstens 1,027 pu).
- Die Stufen folgen dem Tagesgang der Lasten: Im Wohngebiet wird vor allem abends (17 bis 21 Uhr) und am Wochenende auch mittags hochgeschaltet, im Gewerbe an Werktagen tagsüber (8 bis 17 Uhr), nie am Wochenende. Nachts kehren beide Regler in die Neutralstellung zurück. Hochgerechnet auf ein Jahr sind das etwa 400 bis 430 Stufenschritte je Trafo; ein ECOTAP VPD ist für 500 000 Schaltspiele ausgelegt [21].
- **Grenze der rONT-Regelung:** Im MS-Netz bleiben 8 Stunden knapp unter 0,95 pu an Bus 9 bis Bus 11 (niedrigster Wert 0,949 pu). Der rONT kann sie nicht beheben, weil er nur auf sein Ortsnetz wirkt. Was im MS-Netz möglich ist, wird in Teil B untersucht.

### 5.2 Test der Regelrichtung nach unten (Überspannung)

Ohne Erzeuger tritt in der Zeitreihe keine Überspannung auf. Um die Regelrichtung nach unten zu prüfen, wird ein statischer Betriebspunkt gerechnet: Schwachlast (25 %) und Slack-Spannung 1,08 pu statt 1,03 pu, z. B. nach einer Spannungsanhebung im vorgelagerten Netz. Die Tabelle zeigt in jeder Zeile einen Lastfluss: Zeile 0 ohne Eingriff, jede weitere Zeile nach einem Eingriff der Regler.

In [ ]:
net = copy.deepcopy(ausgelegt.net)
net.load[["p_mw", "q_mvar"]] *= 0.25
net.ext_grid["vm_pu"] = 1.08
rg.regle_statisch(net, lv_regler())[["Eingriff", "Stufen", "U min NS [pu]", "U max NS [pu]", "U max MS/NS [pu]"]]

**Ergebnis:** Ohne Eingriff steigen die NS-Spannungen auf bis zu 1,070 pu. Alle drei Regler schalten in einem Schritt eine Stufe nach unten (+1) und bringen das Ortsnetz auf 1,028 bis 1,043 pu zurück ins Band. Die MS-Spannung von 1,074 pu kann der rONT nicht senken, weil er nur auf das Ortsnetz wirkt; dafür wären ein Stufenschalter am 110/20-kV-Trafo oder Blindleistung nötig.

### 5.3 rONT-Regelung in allen Schaltzuständen S1–S3

Wie in Teil 1 wird jeder der acht Schaltzustände (S1, S2, S3 jeweils offen oder geschlossen) als fester Betriebszustand über 12 Wochen gerechnet, jetzt mit den drei rONT-Reglern. So wird geprüft, ob die Regelung auch bei geänderter Topologie des MS-Netzes funktioniert. Oben: ohne Regelung, unten: mit Regelung.

In [ ]:
geregelt_alle = sz.zeitreihe_alle(ausgelegt, prefix="w4_t3r", controllers=lv_regler())

In [ ]:
tab = pd.DataFrame({sz.label(k, kurz=True): pd.concat([band_stunden(s), rg.schaltvorgaenge(s).filter(like="Stufen")])
                    for k, s in geregelt_alle.items()})
tab_u = pd.DataFrame({sz.label(k, kurz=True): band_stunden(s) for k, s in ungeregelt.items()})
display(plots.table(tab_u.astype(float), decimals=3))   # ungeregelt
plots.table(tab.astype(float), decimals=3)              # geregelt

**Ergebnis aller Schaltzustände:**

- In allen acht Schaltzuständen liegt das Ortsnetz mit Regelung immer im Planungsband (niedrigste NS-Spannung 0,950 pu; ohne Regelung 450 bis 473 Stunden darunter).
- Mit geschlossenem S1 sinkt die Spannung an Bus 12 (Anschluss des Ortsnetzes) leicht. Die Regler gleichen das mit etwas mehr Stufenschritten aus (Wohngebiet 96 statt 92); die höchste NS-Spannung steigt auf 1,039 pu, bleibt aber im Band.
- Die verbleibenden Unterschreitungen im MS-Netz (8, 3 bzw. 9 Stunden bei „alle offen“, „S3 zu“ und „S2 zu“) hängen nur vom Schaltzustand ab, nicht von der rONT-Regelung.

---
# Teil B – Backup-Plan für das MS-Netz

## 6 Backup-Plan mit den Trennschaltern S1–S3

Der rONT wirkt nur auf sein Ortsnetz. Für das MS-Netz bleiben als Stellglieder die drei Trennschalter S1–S3, die im Normalbetrieb offen sind. Abschnitt 6 klärt zuerst, wofür solche Trennstellen in der Praxis vorgesehen sind (6.1), ordnet S1–S3 im CIGRE-MV-Netz ein (6.2) und leitet daraus zwei Fälle für den Backup-Plan ab:

- **Fall A – Ausfall eines Betriebsmittels (6.3, 6.4):** Eine Leitung oder ein Trafo fällt aus. Der Backup-Plan legt fest, welche Trennstelle geschlossen wird, damit alle ungestörten Busse weiter versorgt werden. Das ist die Aufgabe, für die Trennstellen nach der Literatur vorgesehen sind.
- **Fall B – Spannungsstützung (6.5):** Die MS-Spannung fällt ohne Ausfall unter eine Schwelle (z. B. bei einer unvorhergesehenen Lastspitze). Der Backup-Plan schließt die Trennstellen stufenweise.

Für beide Fälle gilt: Das Netz ist so ausgelegt (Teil 2 und Teil 3), dass der Backup-Plan im Normalbetrieb **nie** ausgelöst wird.

### 6.1 Was die Literatur zu Trennstellen, Ringen und Kupplungen sagt

Alle Angaben aus Schwab, *Elektroenergiesysteme* [19]:

| Thema | Aussage | Quelle |
|---|---|---|
| Betriebliche Trennstellen | Alle Netze besitzen definierte betriebliche Trennstellen, mit denen einzelne Netzteile für die Beseitigung von Störungen, Wartung und Instandhaltung spannungsfrei geschaltet werden können; jede Schalthandlung ändert die Topologie. | [19, S. 512] |
| Offene Ringe im MS-Netz | Zur Erhöhung der Versorgungssicherheit werden **offene Ringnetze** bevorzugt: Die Enden zweier Strahlen („Halbringe“) können verbunden werden. **Im Normalbetrieb ist die Verbindung geöffnet**, die Ringnetze werden wie Strahlennetze betrieben, was den Netzschutz wesentlich vereinfacht. | [19, S. 521 f., Bild 11.6b] |
| Wiederversorgung nach einer Störung | Bei einer Störung wird das betroffene Betriebsmittel **herausgetrennt** und die betrieblich vorgesehene **Trennstelle geschlossen**. Alle ungestörten Ortsnetzstationen lassen sich dann weiter versorgen. | [19, S. 522] |
| Ablauf im Ringnetz | Im Kurzschlussfall schaltet der Netzschutz den betroffenen Halbring ab; nach manuellem Heraustrennen des fehlerhaften Leitungsstücks und Zuschalten der vorgesehenen Querverbindungen werden die restlichen Verbraucher vom ungestörten Strahl weiterversorgt. | [19, S. 513 f.] |
| Geschlossene Ringe | Permanent geschlossene Ringe verbessern die Spannungshaltung und verringern die Verluste, stellen aber höhere Anforderungen (Schutz, Personal). | [19, S. 514] |
| Ringnetz mit Gegenstation | Kabelstränge aus zwei Umspannwerken münden in einer gemeinsamen Schaltanlage (Gegenstation) mit Sammelschienen-Längstrennung (LT). | [19, S. 522 f., Bild 11.8] |
| Industrienetz | Die Längskupplungen zwischen den Sammelschienenabschnitten sind im Normalbetrieb geöffnet („offen betriebenes Ringnetz“). | [19, S. 526, Bild 11.13] |
| (n-1)-Prinzip | Beim Ausfall eines Betriebsmittels bleibt die Versorgung erhalten, ohne dass ein Betriebsmittel überlastet wird. Das (n-1)-Prinzip muss für die **Jahreshöchstlast** erfüllt sein. Wird es verletzt, muss der (n-1)-Zustand in kürzester Zeit durch geeignete Schalthandlungen wiederhergestellt werden. | [19, S. 6 f.] |
| Parallelbetrieb von Trafos | Voraussetzungen: gleiche Übersetzung und gleiche Phasenlage (sonst dauernder Ausgleichsstrom), etwa gleiche Kurzschlussimpedanzen (Lastaufteilung nach Bemessungsleistung). | [19, S. 396 f.] |
| Maschen | In vermaschten Netzen teilen sich die Ströme nach den Leitungsimpedanzen auf; Leitungen niedriger Impedanz können überlastet werden, Kreisströme erhöhen die Verluste. Der Netzbetrieb steuert u. a. über die **Netztopologie** und die Stufenschalter. | [19, S. 437 f.] |
| Kuppelleitungen | Kuppelleitungen (Verbundnetz) dürfen stationär nur mit einem Teil ihrer Kapazität belastet werden, damit im Störfall Reserve bleibt. | [19, S. 36] |

**Folgerungen für den Backup-Plan:**

1. Die Trennstellen sind in erster Linie für die **Wiederversorgung nach einem Ausfall** vorgesehen (Fall A). Daraus ergibt sich ein fester Schaltplan: Für jedes Betriebsmittel wird vorab festgelegt, welche Trennstelle nach dem Heraustrennen geschlossen wird.
2. Ob das Netz (n-1)-sicher ist, wird bei **Jahreshöchstlast** geprüft.
3. Topologieänderungen sind auch ein Mittel der Spannungshaltung (geschlossener Ring verbessert die Spannung). Im MS-Netz wird der Ring aber wegen des Schutzes offen betrieben; ein Schließen zur Spannungsstützung (Fall B) ist deshalb nur eine vorübergehende Notmaßnahme.
4. Wird über S1 gekuppelt, laufen die beiden 110/20-kV-Trafos parallel. Das ist zulässig, weil sie baugleich sind (25 MVA, gleiche Schaltgruppe und Kurzschlussspannung, keine Stufenschalter).

### 6.2 Die Trennschalter im CIGRE-MV-Netz

| Schalter | Leitung | verbindet | Rolle nach [19] |
|---|---|---|---|
| S2 | Line 6-7 | Ende von Strang 1-2-3-4-5-6 mit Strang 3-8-7 (beide aus Trafo 0-1) | Ringtrennstelle innerhalb von Abgang 1 (Bild 11.6b) |
| S3 | Line 11-4 | Ende von Strang 3-8-9-10-11 mit Strang 3-4 (beide aus Trafo 0-1) | Ringtrennstelle innerhalb von Abgang 1 (Bild 11.6b) |
| S1 | Line 14-8 | Ende von Abgang 2 (Trafo 0-12, Bus 12-13-14) mit Abgang 1 (Bus 8) | Kupplung zweier Abgänge mit eigenem Trafo, vergleichbar einer Gegenstation bzw. Längskupplung (Bild 11.8, 11.13); geschlossen laufen beide Trafos parallel |

Netzplan im Normalschaltzustand (Jahreshöchstlast, alle Trennschalter offen):

In [ ]:
P_netz = ungeregelt["000"].results.ext_grid_p_mw.squeeze()
H_MAX = P_netz.idxmax()                       # Stunde der Jahreshöchstlast (höchste Leistung aus dem 110-kV-Netz)
print(f"Jahreshöchstlast: {H_MAX:%d.%m. %H:%M} Uhr, {P_netz.max():.1f} MW aus dem 110-kV-Netz")

def netz_zur_stunde(h):
    """Ausgelegtes Netz mit den Lasten einer Stunde der Zeitreihe (statischer Lastfluss)."""
    r = ungeregelt["000"].results
    net = copy.deepcopy(ausgelegt.net)
    net.load["p_mw"] = net.load["name"].map(r.load_p_mw.loc[h]).to_numpy()
    net.load["q_mvar"] = net.load["name"].map(r.load_q_mvar.loc[h]).to_numpy()
    return net

spitzennetz = netz_zur_stunde(H_MAX)
n = copy.deepcopy(spitzennetz)
pp.runpp(n)
_ = netplan.plot_static(n, title="Normalschaltzustand bei Jahreshöchstlast")

### 6.3 Fall A: Wiederversorgung nach dem Ausfall einer Leitung ((n-1)-Prüfung)

**Was hier passiert, Schritt für Schritt:**

1. **Betriebspunkt:** Lasten der Stunde mit der Jahreshöchstlast (siehe oben), denn dort muss das (n-1)-Prinzip erfüllt sein [19, S. 6 f.]. Alle Trennschalter sind offen, die Stufenschalter stehen in Neutralstellung (die rONT-Regelung kommt erst in Teil C dazu).
2. **Ausfall:** Eine MS-Leitung wird außer Betrieb genommen (`in_service = False`). Das entspricht dem Zustand, nachdem der Schutz ausgelöst und das fehlerhafte Leitungsstück herausgetrennt wurde. Untersucht werden alle 12 MS-Leitungen, die im Normalbetrieb Leistung führen (die drei Leitungen mit S1–S3 sind ohnehin offen).
3. **Maßnahmen:** Für jeden Ausfall werden alle acht Schaltoptionen gerechnet: keine Maßnahme, ein Schalter, zwei Schalter, alle drei.
4. **Bewertung:** Eine Maßnahme ist **zulässig**, wenn danach alle Busse wieder versorgt sind, keine MS-Spannung unter 0,90 pu liegt und keine MS-Leitung und kein 110/20-kV-Trafo über 100 % ausgelastet ist.
5. **Schaltplan:** Je Ausfall wird die erste zulässige Maßnahme in dieser Reihenfolge gewählt: S2, S3, S1, dann Kombinationen. S2 und S3 haben Vorrang, weil sie nur innerhalb von Abgang 1 schalten; S1 kommt erst danach, weil es beide Trafos parallel schaltet.

In der Übersicht steht je Ausfall und Maßnahme, ob die Maßnahme zulässig ist (✔), wie viele Busse (MS und NS) ohne Versorgung bleiben oder ob eine Grenze verletzt ist:

In [ ]:
AUSFAELLE_LEITUNG = [n for n in ausgelegt.net.line["name"] if n.startswith("Line") and n not in ("Line 6-7", "Line 11-4", "Line 14-8")]
OPTIONEN = [(), ("S2",), ("S3",), ("S1",), ("S2", "S3"), ("S2", "S1"), ("S3", "S1"), ("S2", "S3", "S1")]

analyse = rg.ausfall_analyse(spitzennetz, AUSFAELLE_LEITUNG, OPTIONEN, g["MS"])

def status(row):
    if row["Bemerkung"]:
        return "✘ " + row["Bemerkung"]
    if row["nicht versorgte Busse"] > 0:
        k = int(row["nicht versorgte Busse"])
        return f"{k} Bus unversorgt" if k == 1 else f"{k} Busse unversorgt"
    return "✔" if row["zulässig"] else "✘ Grenze verletzt"

uebersicht = analyse.assign(Status=analyse.apply(status, axis=1)).pivot(index="Ausfall", columns="Maßnahme", values="Status")
uebersicht.loc[AUSFAELLE_LEITUNG, [m for m in analyse["Maßnahme"].unique()]]

**Wiederversorgungsplan:** je Ausfall die gewählte Maßnahme und der Netzzustand danach:

In [ ]:
plan_tab = rg.wiederversorgungsplan(analyse)
display(plots.table(plan_tab, decimals=3, int_cols=["ohne Maßnahme nicht versorgt"]))
WIEDERVERSORGUNG = {a: [s.split()[0] for s in m.split(" + ")] for a, m in plan_tab["Maßnahme"].items() if "schließen" in m}
WIEDERVERSORGUNG

**Beispiel Ausfall Line 2-3:** oben nach dem Heraustrennen (Bus 3 bis Bus 11 ohne Versorgung, grau), unten nach dem Schließen von S1. Abgang 1 wird jetzt rückwärts über Bus 12-13-14 und S1 aus Trafo 0-12 versorgt.

In [ ]:
for schalter, titel in (([], "Ausfall Line 2-3 – vor der Wiederversorgung"),
                       (["S1"], "Ausfall Line 2-3 – nach dem Schließen von S1")):
    n = copy.deepcopy(spitzennetz)
    n.line.loc[n.line["name"] == "Line 2-3", "in_service"] = False
    n.switch.loc[n.switch["name"].isin(schalter), "closed"] = True
    pp.runpp(n)
    _ = netplan.plot_static(n, title=titel)

**Ergebnis der (n-1)-Prüfung für die Leitungen:**

- **Ohne Maßnahme** sind nach jedem Leitungsausfall 1 bis 10 Busse ohne Versorgung (bei Line 1-2 der ganze Abgang hinter Bus 1).
- **Mit Maßnahme** lässt sich jeder der 12 Leitungsausfälle bei Jahreshöchstlast vollständig wiederversorgen, ohne eine Grenze zu verletzen. Das Netz ist für Leitungsausfälle also (n-1)-sicher, wenn die Trennstellen nach Plan geschlossen werden.
- **Wiederversorgungsplan:**
  - S2 schließen bei Ausfall von Line 3-4, 4-5, 5-6, 7-8 und 3-8,
  - S3 schließen bei Ausfall von Line 8-9, 9-10 und 10-11,
  - S1 schließen bei Ausfall von Line 1-2, 2-3, 12-13 und 13-14.
- Welche Trennstelle passt, ergibt sich aus der Topologie: Fällt eine Leitung in einem Strang von Abgang 1 aus, wird der abgetrennte Teil über die Trennstelle am Strangende vom anderen Strang desselben Abgangs versorgt (S2 oder S3). Fällt eine Leitung am Anfang von Abgang 1 (Line 1-2, 2-3) oder in Abgang 2 aus, hilft nur die Kupplung über S1 aus dem anderen Trafo.
- **Netzzustand danach:** Die niedrigste MS-Spannung liegt zwischen 0,928 und 0,951 pu (über der Arbeitsgrenze 0,90 pu, teils unter dem Planungsband). Die höchste Leitungsauslastung steigt auf höchstens 86 %, der stärker belastete 110/20-kV-Trafo auf höchstens 87 %. In den S1-Fällen sinkt die NS-Spannung im Ortsnetz auf 0,915 pu; das gleicht die rONT-Regelung aus (Teil C).
- **Beispiel Line 2-3:** Nach dem Schließen von S1 fließt die Leistung für Bus 3 bis Bus 11 aus Trafo 0-12 über Bus 12, 13, 14 und S1 zu Bus 8 und von dort weiter durch Abgang 1; die Flussrichtung auf Line 3-8 kehrt sich um. Trafo 0-12 übernimmt die zusätzliche Last (87 %), Trafo 0-1 versorgt nur noch Bus 1 und Bus 2.

### 6.4 Fall A: Ausfall eines 110/20-kV-Transformators

Fällt Trafo 0-1 oder Trafo 0-12 aus, ist der ganze Abgang ohne Versorgung. Die einzige Verbindung zum anderen Trafo ist S1 am Ende der Abgänge (Bus 8 – Bus 14). Die (n-1)-Prüfung bei Jahreshöchstlast:

In [ ]:
analyse_trafo = rg.ausfall_analyse(spitzennetz, ["Trafo 0-1", "Trafo 0-12"], [(), ("S1",), ("S2", "S3", "S1")], g["MS"])
analyse_trafo.round(3)

Bei Jahreshöchstlast konvergiert der Lastfluss nicht, das Netz kann die Leistung über S1 nicht übertragen. Bis zu welcher Last wäre eine Wiederversorgung über S1 möglich? Dazu werden alle Lasten der Spitzenstunde schrittweise verkleinert:

In [ ]:
zeilen = []
for trafo in ("Trafo 0-1", "Trafo 0-12"):
    for f in np.arange(0.15, 0.55, 0.05):
        n = copy.deepcopy(spitzennetz)
        n.load[["p_mw", "q_mvar"]] *= f
        n.trafo.loc[n.trafo["name"] == trafo, "in_service"] = False
        n.switch.loc[n.switch["name"] == "S1", "closed"] = True
        z = {"Ausfall": trafo, "Last [% der Spitzenstunde]": round(f * 100), "Last [MW]": n.load.p_mw.sum()}
        try:
            pp.runpp(n)
            mv = n.bus.index[n.bus["name"].isin(g["MS"])]
            ll = n.res_line.loading_percent[n.line.from_bus.isin(mv) & n.line.in_service]
            z.update({"Trafo max [%]": n.res_trafo.loading_percent[n.trafo.lv_bus.isin(mv)].max(),
                      "U min MS [pu]": n.res_bus.vm_pu[mv].min(), "Leitung max MS [%]": ll.max(),
                      "am stärksten belastet": n.line.at[ll.idxmax(), "name"]})
        except pp.LoadflowNotConverged:
            z["am stärksten belastet"] = "Lastfluss konvergiert nicht"
        zeilen.append(z)
trafo_scan = pd.DataFrame(zeilen)
display(plots.table(trafo_scan, decimals=2))
print(f"Kleinste Last der Zeitreihe: {P_netz.min():.1f} MW ({P_netz.min() / P_netz.max():.0%} der Jahreshöchstlast)")

**Vergleich: Kupplung direkt im Umspannwerk.** In einem realen Umspannwerk mit zwei Trafos würde man die beiden 20-kV-Sammelschienen über eine Sammelschienen-Längskupplung verbinden [19, S. 522 f., Bild 11.8]. Im CIGRE-Modell gibt es diese Kupplung nicht. Hypothetisch wird sie als Schalter zwischen Bus 1 und Bus 12 eingefügt; damit wird geprüft, ob der verbleibende Trafo die Last allein tragen könnte. Zusätzlich die Zahl der Stunden der Zeitreihe, in denen die Scheinleistung aus dem 110-kV-Netz unter der Bemessungsleistung eines Trafos (25 MVA) liegt:

In [ ]:
zeilen = []
for trafo in ("Trafo 0-1", "Trafo 0-12"):
    n = copy.deepcopy(spitzennetz)
    b1, b12 = (int(n.bus.index[n.bus["name"] == b][0]) for b in ("Bus 1", "Bus 12"))
    pp.create_switch(n, b1, b12, et="b", closed=True, name="LT (hypothetisch)")
    n.trafo.loc[n.trafo["name"] == trafo, "in_service"] = False
    pp.runpp(n)
    mv = n.bus.index[n.bus["name"].isin(g["MS"])]
    zeilen.append({"Ausfall": trafo, "verbleibender Trafo [%]": n.res_trafo.loading_percent[n.trafo.lv_bus.isin(mv)].max(),
                   "U min MS [pu]": n.res_bus.vm_pu[mv].min(),
                   "Leitung max MS [%]": n.res_line.loading_percent[n.line.from_bus.isin(mv)].max()})
display(plots.table(pd.DataFrame(zeilen).set_index("Ausfall"), decimals=3))

r0 = ungeregelt["000"].results
S_netz = np.hypot(r0.ext_grid_p_mw.squeeze(), r0.ext_grid_q_mvar.squeeze())
sn = float(ausgelegt.net.trafo.loc[ausgelegt.net.trafo["name"] == "Trafo 0-1", "sn_mva"].iloc[0])
print(f"Scheinleistung aus dem 110-kV-Netz: max {S_netz.max():.1f} MVA; "
      f"Stunden ≤ {sn:.0f} MVA: {int((S_netz <= sn).sum())} von {len(S_netz)} ({(S_netz <= sn).mean():.0%})")

**Ergebnis Trafoausfall:**

- **Über S1 nicht wiederversorgbar:** Bei Jahreshöchstlast konvergiert der Lastfluss nicht. Ursache ist nicht der verbleibende Trafo, sondern der lange Weg über die Leitungen: Die ganze Leistung des ausgefallenen Abgangs müsste über S1 am Ende der Abgänge und dann durch die schwach ausgelegten Leitungen von Abgang 1 (zuerst Line 3-8 bzw. Line 2-3) fließen. Schon bei 25 % der Spitzenlast sind diese Leitungen überlastet und die MS-Spannung fällt unter 0,90 pu. Zulässig wäre die Wiederversorgung nur bis etwa 20 % der Spitzenlast (rund 7,5 MW); die kleinste Last der ganzen Zeitreihe liegt mit 10,4 MW (28 %) darüber.
- Das bestätigt die Literatur: Eine Kupplung am Strangende ist für die Wiederversorgung einzelner Leitungsabschnitte gedacht, nicht als Reserve für einen ganzen Trafo; Kuppelleitungen tragen nur einen Teil der Übertragungsfähigkeit [19, S. 36].
- **Mit Längskupplung im Umspannwerk (hypothetisch):** Die Leitungen wären nicht mehr der Engpass (höchstens 82 %), aber der verbleibende Trafo wäre bei Jahreshöchstlast mit 159 % überlastet. Ein Trafo allein reicht in 59 % der Stunden (Scheinleistung bis 25 MVA). Für die übrigen Stunden wäre eine zeitlich begrenzte Notüberlastung nach DIN IEC 60076-7 zu prüfen, oder die Trafos müssten so ausgelegt werden, dass einer die Höchstlast allein tragen kann (vgl. [19, S. 515]).
- **Folgerung für den Backup-Plan:** Der Plan deckt Leitungsausfälle ab, aber keinen Ausfall eines 110/20-kV-Trafos. Für den Trafoausfall bräuchte das Netz eine Sammelschienen-Längskupplung im Umspannwerk und größere Trafos. Das ist eine Netzausbaumaßnahme und gehört zum Ausblick.

### 6.5 Fall B: Spannungsstützung ohne Ausfall

**Idee:** Fällt die MS-Spannung ohne Ausfall eines Betriebsmittels unter eine Schwelle, werden die Ringe vorübergehend geschlossen. Ein geschlossener Ring verbessert die Spannungshaltung [19, S. 514]; da die Ringe im MS-Netz wegen des Schutzes offen betrieben werden [19, S. 521 f.], ist das nur eine Notmaßnahme.

**Wirkung der einzelnen Schalter:** Mittlere Änderung der Spannung je MS-Bus gegenüber „alle offen“ in den 20 Stunden mit der niedrigsten MS-Spannung (Zeitreihen aus Teil 3, ohne Regelung), in Prozentpunkten:

In [ ]:
base = ungeregelt["000"].results.vm_pu[g["MS"]]
schwach_h = base.min(axis=1).nsmallest(20).index
wirkung = pd.DataFrame({sz.label(k, kurz=True): ((s.results.vm_pu[g["MS"]] - base).loc[schwach_h].mean() * 100)
                        for k, s in ungeregelt.items()})
plots.table(wirkung, decimals=2, cmap="RdYlGn", vmin=-2.5, vmax=2.5)

**Was die Tabelle zeigt:**

- **S2 allein wirkt praktisch nicht.** Bus 6 und Bus 7 haben fast dieselbe Spannung, über die geschlossene Masche fließt kaum Leistung.
- **S3 hebt Bus 7 bis Bus 11 um 0,15 bis 0,3 Prozentpunkte** an, senkt aber Bus 4 bis Bus 6 leicht. **S2 und S3 zusammen** heben Bus 7 bis Bus 11 an und beseitigen die 8 Stunden unter dem Planungsband in Abgang 1.
- **S1 hebt Abgang 1 um gut 2 Prozentpunkte**, senkt aber Abgang 2: Bus 13 um 1,3 und Bus 14 um 2,0 Prozentpunkte, Bus 12 (Anschluss des Ortsnetzes) um 0,2 Prozentpunkte. Über S1 übernimmt Trafo 0-12 einen Teil der Last von Abgang 1.
- **Größenordnung:** S2 und S3 wirken mit weniger als 0,5 Prozentpunkten und können nur kleine Unterschreitungen ausgleichen. S1 wirkt stärker, koppelt aber beide Trafos. Daraus ergibt sich der Stufenplan unten: zuerst S2 und S3, S1 erst im Notfall.

**Stufenplan:**

| Stufe | Maßnahme | Auslösung (niedrigste MS-Spannung) | Rücknahme | Begründung |
|---|---|---|---|---|
| 1 | S2 und S3 schließen | < 0,94 pu | ≥ 0,95 pu | schließt nur die Ringe innerhalb von Abgang 1, beide Trafos bleiben getrennt |
| 2 | S1 schließen | < 0,92 pu | ≥ 0,96 pu | kuppelt beide Abgänge (Parallelbetrieb der Trafos), hebt Abgang 1 deutlich an, senkt aber Abgang 2; nur als Notfallmaßnahme |

- Die Auslöseschwellen liegen bewusst **unter** dem Planungsband: Stufe 1 bei mehr als 1 Prozentpunkt Unterschreitung, Stufe 2, wenn nur noch 2 Prozentpunkte Abstand zur Arbeitsgrenze 0,90 pu bleiben.
- Die Rücknahmeschwellen liegen um die Wirkung der Stufe plus Reserve höher, damit nach dem Öffnen nicht sofort wieder geschlossen wird (Hysterese).
- Je Prüfung wird höchstens eine Stufe zu- oder abgeschaltet; Rücknahme in umgekehrter Reihenfolge.
- Umsetzung: Klasse `SchaltPlan` in `regelung.py`.

In [ ]:
def backup_plan(wiederversorgung=None):
    return rg.SchaltPlan(g["MS"], [(["S2", "S3"], 0.94, 0.95), (["S1"], 0.92, 0.96)], wiederversorgung=wiederversorgung)

**Test:** In der Zeitreihe fällt die MS-Spannung nie unter 0,94 pu (niedrigster Wert 0,949 pu), der Plan würde also nie auslösen. Damit trotzdem sichtbar wird, was er tut, wird er an drei statischen Lastspitzen getestet: alle Lasten gleichzeitig mit 100 %, 110 % und 120 % ihrer Bemessungsleistung. Hier ist **nur** der Backup-Plan aktiv (keine rONT-Regelung), damit seine Wirkung allein zu sehen ist.

So ist die Tabelle zu lesen: Für jede Lastspitze wird zuerst ein Lastfluss ohne Eingriff gerechnet (Zeile „Start“). Liegt die niedrigste MS-Spannung unter der Auslöseschwelle der nächsten Stufe, schließt der Backup-Plan die Schalter dieser Stufe, und der Lastfluss wird neu gerechnet (nächste Zeile). Das wiederholt sich, bis keine Schwelle mehr unterschritten ist.

In [ ]:
zeilen = []
for faktor in (1.0, 1.1, 1.2):
    net = copy.deepcopy(ausgelegt.net)
    net.load[["p_mw", "q_mvar"]] *= faktor
    prot = rg.regle_statisch(net, [backup_plan()])       # Protokoll: Start und je Eingriff eine Zeile
    for it, z in prot.iterrows():
        # Lastfluss des Schritts nachrechnen, um MS-Werte getrennt auszuweisen
        n = copy.deepcopy(ausgelegt.net)
        n.load[["p_mw", "q_mvar"]] *= faktor
        zu = [] if z["geschlossene Trennschalter"] == "keine" else z["geschlossene Trennschalter"].split(", ")
        n.switch.loc[n.switch["name"].isin(zu), "closed"] = True
        pp.runpp(n)
        mv = n.bus.index[n.bus["name"].isin(g["MS"])]
        zeilen.append({"Lastspitze": f"{faktor:.0%}", "Schritt": "Start" if it == 0 else f"Eingriff {it}",
                       "geschlossene Trennschalter": z["geschlossene Trennschalter"],
                       "U min MS [pu]": n.res_bus.vm_pu[mv].min(),
                       "schwächster MS-Bus": n.bus.at[n.res_bus.vm_pu[mv].idxmin(), "name"],
                       "U Bus 12 [pu]": n.res_bus.vm_pu[n.bus.index[n.bus["name"] == "Bus 12"][0]],
                       "Leitung max MS [%]": n.res_line.loading_percent[n.line.from_bus.isin(mv)].max(),
                       "Trafo max HS/MS [%]": n.res_trafo.loading_percent[n.trafo.lv_bus.isin(mv)].max()})
stuetzung = pd.DataFrame(zeilen).set_index(["Lastspitze", "Schritt"])
plots.table(stuetzung, decimals=3)

**So läuft der Backup-Plan bei den drei Lastspitzen ab:**

| Lastspitze | Start: U min MS | Stufe 1 (S2 + S3) | Stufe 2 (S1) | Ergebnis |
|---|---|---|---|---|
| 100 % | 0,923 pu (Bus 11) | ausgelöst (< 0,94 pu) → 0,926 pu | nicht nötig (≥ 0,92 pu) | über der Arbeitsgrenze, unter dem Planungsband |
| 110 % | 0,909 pu (Bus 11) | ausgelöst → 0,913 pu | ausgelöst (< 0,92 pu) → **0,948 pu** | knapp unter dem Planungsband |
| 120 % | 0,895 pu (Bus 11, **unter 0,90 pu**) | ausgelöst → 0,899 pu | ausgelöst → **0,939 pu** | Arbeitsgrenze wieder eingehalten |

- **Stufe 1** hebt die Spannung nur um 0,3 bis 0,4 Prozentpunkte. Der schwächste Bus wandert dabei von Bus 11 zu Bus 6, weil S3 Bus 4 bis Bus 6 leicht absenkt (siehe Tabelle oben).
- **Stufe 2** wirkt stark: Über S1 übernimmt Trafo 0-12 einen Teil der Last von Abgang 1. Die MS-Spannung steigt um 3,5 bis 4 Prozentpunkte, und zusätzlich sinkt die höchste Leitungsauslastung von 108 % auf 59 % (110 %) bzw. von 120 % auf 65 % (120 %).
- **Grenzen:** Trafo 0-1 bleibt bei 110 % und 120 % Last über 100 % ausgelastet (107 % bzw. 117 %), weil er Abgang 1 weiterhin zum großen Teil versorgt. Bus 12 sinkt durch S1 nur geringfügig (um 0,3 bis 0,4 Prozentpunkte).
- Der Test zeigt, dass der Backup-Plan im Notfall eine Verletzung der Arbeitsgrenze 0,90 pu verhindern kann. Er ersetzt aber keinen Netzausbau: Die Überlastung des Trafos bleibt bestehen.

---
# Teil C – Zusammenspiel

## 7 Simulation mit rONT-Regelung und Backup-Plan gleichzeitig

Jetzt arbeiten alle Regler gemeinsam in der Zeitreihe (12 Wochen, stündlich):

| Regler | wirkt auf | greift ein, wenn … |
|---|---|---|
| 3 × `TapRegler` (rONT) | Stufenschalter der Ortsnetztrafos | eine NS-Spannung des eigenen Teilnetzes das Band 0,95 … 1,05 pu verlässt (Teil A) |
| `Wiederversorgung` | S1–S3 nach dem Wiederversorgungsplan aus 6.3 | ein Betriebsmittel ausgefallen ist (Fall A) |
| `SchaltPlan` | S2+S3, dann S1 | die niedrigste MS-Spannung unter 0,94 bzw. 0,92 pu fällt (Fall B) |

Nach jedem Lastfluss prüfen alle Regler das Ergebnis; greift einer ein, wird dieselbe Stunde neu gerechnet. Schalter, die die Wiederversorgung geschlossen hat, öffnet der `SchaltPlan` nicht wieder, solange das Betriebsmittel ausgefallen ist.

Drei Fälle:

1. **Normalbetrieb:** gleiche Randbedingungen wie in Abschnitt 5.1. Erwartung: Der Backup-Plan greift nie ein, das Ergebnis ist identisch mit der reinen rONT-Regelung.
2. **Störfall:** Line 2-3 fällt aus und bleibt für die ganzen 12 Wochen außer Betrieb (ungünstigster Fall: die Leitung, nach deren Ausfall die meisten Busse über S1 versorgt werden müssen).
3. **Laststeigerung:** alle Lasten 20 % bzw. 30 % höher als geplant, z. B. durch Wärmepumpen und Elektromobilität ohne Netzausbau.

### 7.1 Normalbetrieb

In [ ]:
def alle_regler():
    wv = rg.Wiederversorgung(WIEDERVERSORGUNG)
    return lv_regler() + [wv, backup_plan(wiederversorgung=wv)]

kombi = {}
kombi["Normalbetrieb"] = ausgelegt.copy("w4_t3c_normal", overwrite=True)
kombi["Normalbetrieb"].runSimulation(controllers=alle_regler())

In [ ]:
def kombi_tabelle(sims):
    rows = {}
    for name, s in sims.items():
        sw = s.results.switch_closed[["S1", "S2", "S3"]]
        rows[name] = pd.concat([band_stunden(s),
                                pd.Series({"Leitung max MS [%]": s.results.line_loading[[c for c in s.results.line_loading if c.startswith("Line")]].max().max(),
                                           "Trafo max HS/MS [%]": s.results.trafo_loading[["Trafo 0-1", "Trafo 0-12"]].max().max()}),
                                rg.schaltvorgaenge(s),
                                sw.sum().rename(lambda c: f"Stunden {c} geschlossen")])
    return pd.DataFrame(rows)

plots.table(kombi_tabelle({"nur rONT (5.1)": geregelt, **kombi}).astype(float), decimals=3)

**Ergebnis Normalbetrieb:** Die Ergebnisse sind in allen Kennwerten identisch mit der reinen rONT-Regelung aus Abschnitt 5.1. Die niedrigste MS-Spannung (0,949 pu) liegt über der Auslöseschwelle 0,94 pu, es fällt kein Betriebsmittel aus; der Backup-Plan wird in keiner Stunde ausgelöst (0 Schaltungen). Damit ist die Forderung erfüllt, dass das Netz im Normalbetrieb ohne den Backup-Plan auskommt.

### 7.2 Störfall: Ausfall von Line 2-3

Vor dem ersten Lastfluss meldet die Simulation die nicht versorgten Busse; das ist der Zustand direkt nach dem Ausfall, bevor die Wiederversorgung S1 schließt.

In [ ]:
kombi["Ausfall Line 2-3"] = ausgelegt.copy("w4_t3c_ausfall_2_3", overwrite=True)
kombi["Ausfall Line 2-3"].changeParameter("Line 2-3", "in_service", False)
kombi["Ausfall Line 2-3"].runSimulation(controllers=alle_regler())

In [ ]:
plots.table(kombi_tabelle({"Normalbetrieb": kombi["Normalbetrieb"], "Ausfall Line 2-3": kombi["Ausfall Line 2-3"]}).astype(float), decimals=3)

In [ ]:
s = kombi["Ausfall Line 2-3"]
idx = s.results.info.index[s.results.info.month == "JANUARY"]
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1, 1]})
vm = s.results.vm_pu.loc[idx]
axes[0].plot(idx, vm[g["MS"]].min(axis=1), color="black", label="U min MS")
axes[0].plot(idx, vm[g["NS"]].min(axis=1), color="tab:blue", label="U min NS")
axes[0].plot(idx, vm[g["NS"]].max(axis=1), color="tab:blue", ls="--", label="U max NS")
for v, c, ls in ((V_MIN, "red", "--"), (V_PLAN_MIN, "orange", ":"), (V_PLAN_MAX, "orange", ":"), (0.94, "grey", "-."), (0.92, "grey", "-.")):
    axes[0].axhline(v, color=c, ls=ls, lw=1)
axes[0].set_ylabel("Spannung [pu]")
axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
axes[0].set_title("Ausfall Line 2-3 – Januar (graue Linien: Auslöseschwellen des Backup-Plans)", pad=22)
sw = s.results.switch_closed.loc[idx, ["S1", "S2", "S3"]].astype(bool)
for i, (c, farbe) in enumerate(zip(sw.columns, ("tab:red", "tab:green", "tab:purple"))):
    axes[1].fill_between(idx, i + 0.1, i + 0.9, where=sw[c].to_numpy(), step="post", color=farbe, alpha=0.7)
axes[1].set_ylim(0, 3)
axes[1].set_yticks([0.5, 1.5, 2.5], ["S1", "S2", "S3"])
axes[1].set_ylabel("Balken = geschlossen")
taps = s.results.tap_pos.loc[idx, [TRAFO[x] for x in "RIC"]]
for x, farbe in zip("RIC", ("tab:blue", "tab:green", "tab:orange")):
    axes[2].step(idx, taps[TRAFO[x]], where="post", color=farbe, label=TRAFO[x])
axes[2].set_ylabel("Stufe")
axes[2].set_yticks(range(-2, 3))
axes[2].legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plots.week_axis(axes[2], idx)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
s.createDiagram(Diagram.NETWORK, time=["min", "max_loading"], title="Ausfall Line 2-3 mit Wiederversorgung und Regelung")

**Ergebnis Störfall Line 2-3:**

- **Wiederversorgung:** In der ersten Stunde sind Bus 3 bis Bus 11 ohne Versorgung. Die `Wiederversorgung` schließt S1 sofort (eine Schaltung), danach sind in allen 2016 Stunden alle Busse versorgt. Keine Leitung und kein Trafo ist überlastet (höchstens 65 % bzw. 87 %).
- **MS-Spannung:** Weil Abgang 1 jetzt über den langen Weg Bus 12–13–14–8 versorgt wird, fällt die MS-Spannung öfter unter das Planungsband (257 statt 8 Stunden, niedrigster Wert 0,932 pu an Bus 5). Die Arbeitsgrenze 0,90 pu wird nie verletzt.
- **Spannungsstützung:** In 114 Stunden (abends 17 bis 20 Uhr und mittags, nur in den Wintermonaten) fällt die MS-Spannung unter 0,94 pu, und der `SchaltPlan` schließt zusätzlich S2 und S3 (76 Schaltungen je Schalter). Die Wirkung ist klein, weil S1 schon geschlossen ist und S3 den schwächsten Bereich (Bus 4 bis Bus 6) eher absenkt; in 62 Stunden bleibt die Spannung unter 0,94 pu. S1 bleibt die ganze Zeit zu, weil die Wiederversorgung es gesperrt hat.
- **Ortsnetz:** Die rONT-Regler halten das Ortsnetz trotz des Störfalls in jeder Stunde im Planungsband (0,950 bis 1,039 pu). Sie brauchen dafür mehr Stufenschritte (Wohngebiet 132 statt 92, Gewerbe 120 statt 100, Industrie 10 statt 0), weil Bus 12 durch die zusätzliche Last über Trafo 0-12 etwas tiefer liegt. Im Wohngebiet wird abends erstmals auch die zweite Stufe (−2) genutzt; der Stellbereich ist damit zum Teil ausgeschöpft.
- **Zusammenspiel:** Die drei Regler stören sich nicht: Je Stunde reicht eine Nachrechnung, und der `SchaltPlan` öffnet den von der Wiederversorgung geschlossenen Schalter S1 nicht.

### 7.3 Laststeigerung um 20 % und 30 %

Alle Lasten (Jahresenergie und Höchstleistung) werden um 20 % bzw. 30 % erhöht; das Netz bleibt unverändert, kein Ausfall.

In [ ]:
for faktor in (1.2, 1.3):
    name = f"Last +{faktor - 1:.0%}"
    s = ausgelegt.copy(f"w4_t3c_last{round(faktor * 100)}", overwrite=True)
    s.net.load[["annual_kwh", "p_mw", "q_mvar"]] *= faktor
    s.save(results=False)
    s.runSimulation(controllers=alle_regler())
    kombi[name] = s
plots.table(kombi_tabelle({k: kombi[k] for k in ("Normalbetrieb", "Last +20%", "Last +30%")}).astype(float), decimals=3)

In [ ]:
sim.compare([kombi["Normalbetrieb"], kombi["Last +20%"], kombi["Last +30%"]], Diagram.VOLTAGE, ["Bus 11"], months=Month.JANUARY,
            markers=PLAN_MARKER, title="Bus 11 (schwächster MS-Bus) – Normalbetrieb und Laststeigerung")

**Ergebnis Laststeigerung:**

| Kennwert | Normalbetrieb | Last +20 % | Last +30 % |
|---|---|---|---|
| Stunden MS < 0,95 pu | 8 | 197 | 409 |
| U min MS | 0,949 pu | 0,931 pu | 0,920 pu |
| Stunden NS < 0,95 pu | 0 | 0 | 3 |
| Leitung max MS | 79 % | 96 % | **106 %** (33 Stunden > 100 %) |
| Trafo 0-1 max | 83 % | **100,3 %** (1 Stunde) | **109 %** (70 Stunden > 100 %) |
| Stunden S2 + S3 geschlossen | 0 | 106 | 224 |
| Stunden S1 geschlossen | 0 | 0 | 0 |

- **Ortsnetz:** Bei +20 % halten die rONT-Regler das Ortsnetz noch vollständig im Band, mit mehr als doppelt so vielen Stufenschritten. Bei +30 % reichen die zwei Stufen in 3 Stunden nicht mehr aus (niedrigster Wert 0,945 pu); der Stellbereich ist erschöpft.
- **MS-Netz:** Stufe 1 des Backup-Plans (S2 + S3) wird in 106 bzw. 224 Stunden ausgelöst, vor allem abends im Winter. Stufe 2 (S1) wird nie ausgelöst, weil die MS-Spannung nicht unter 0,92 pu fällt (niedrigster Wert genau 0,920 pu). Die Arbeitsgrenze 0,90 pu wird nie verletzt.
- **Neue Schwachstelle – Überlastung:** Bei +30 % sind Trafo 0-1 in 70 Stunden und die höchstbelastete MS-Leitung (Line 2-3) in 33 Stunden über 100 % ausgelastet. Der Backup-Plan reagiert nur auf die Spannung, nicht auf die Auslastung. Der statische Test in 6.5 zeigt, dass S1 die Leitungsauslastung deutlich senken würde (von 108 % auf 59 %), den Trafo aber kaum entlastet.
- **Folgerung:** Bis etwa +20 % Last reichen rONT-Regelung und Backup-Plan. Darüber ist Netzausbau nötig (stärkerer Trafo 0-1, Verstärkung von Abgang 1, siehe auch Teil 6 zu Line 3-8). Der Backup-Plan ließe sich um ein Auslastungskriterium erweitern (S1 schließen, wenn eine MS-Leitung über 100 % ausgelastet ist), aber auch das wäre nur eine Notmaßnahme.

## 8 Fazit

**Teil A – rONT-Regelung:**

- Drei unabhängige rONT-Regler mit Messung im eigenen Teilnetz halten das Ortsnetz in allen Schaltzuständen im Planungsband 0,95 bis 1,05 pu (vorher 450 Stunden darunter). Fast immer reicht eine Stufe; im Jahr sind etwa 400 bis 430 Stufenschritte je Trafo zu erwarten.
- Entscheidend ist die Messung an den Strangenden; an der Sammelschiene wäre keine Verletzung zu sehen gewesen.
- Eine feste Stufenstellung reicht nicht: Eine Stellung, die die Spitzenlast abdeckt, erzeugt bei Schwachlast Überspannung (bis 1,073 pu). Die Regelrichtung nach unten funktioniert (Überspannungstest).

**Teil B – Backup-Plan:**

- Nach der Literatur sind Trennstellen im MS-Netz vor allem für die Wiederversorgung nach einem Ausfall vorgesehen; im Normalbetrieb sind die Ringe offen [19, S. 521 f.]. Der Backup-Plan hat deshalb zwei Teile: Wiederversorgung (Fall A) und Spannungsstützung (Fall B).
- **Fall A:** Alle 12 Leitungsausfälle lassen sich bei Jahreshöchstlast mit genau einem Schalter vollständig wiederversorgen (S2, S3 oder S1). Für Leitungsausfälle ist das Netz damit (n-1)-sicher. Einen Ausfall eines 110/20-kV-Trafos kann S1 nicht auffangen; dafür wären eine Längskupplung im Umspannwerk und größere Trafos nötig.
- **Fall B:** Die Spannungsstützung (S2 + S3, dann S1) verhindert bei Lastspitzen von 110 % und 120 % eine Verletzung der Arbeitsgrenze. S2 und S3 wirken nur schwach, S1 stark.

**Teil C – Zusammenspiel:**

- Im Normalbetrieb greift der Backup-Plan nie ein; die Ergebnisse sind identisch mit der reinen rONT-Regelung.
- Beim Ausfall von Line 2-3 versorgt die Wiederversorgung über S1 alle Busse wieder, die rONT-Regler halten das Ortsnetz weiter im Band. Die Regler arbeiten ohne gegenseitige Störung zusammen.
- Bei einer Laststeigerung um 30 % reichen weder der Stellbereich des rONT noch der Backup-Plan; Trafo 0-1 und Line 2-3 werden überlastet. Ab hier ist Netzausbau nötig.

**Grenzen der Untersuchung:** Stundenwerte, symmetrischer Lastfluss, keine Schutztechnik und keine Schaltzeiten; die Wiederversorgung ist idealisiert (sofort nach dem Ausfall). Erzeugung und Leistungsflussumkehr werden erst mit den Szenarien in Teil 4 geprüft.

### Quellen

[15] DIN EN 50160, „Merkmale der Spannung in öffentlichen Elektrizitätsversorgungsnetzen“ (EN 50160:2010 mit Änderungen), Berlin: Beuth.

[19] A. J. Schwab, *Elektroenergiesysteme: Erzeugung, Transport, Übertragung und Verteilung elektrischer Energie*, 3. Aufl. Berlin, Heidelberg: Springer, 2012, doi: 10.1007/978-3-642-21958-0. Verwendet: Kap. 1 (Versorgungssicherheit, (n-1)-Prinzip, S. 6 f.), Kap. 2 (Kuppelleitungen, S. 36), Kap. 9.3.2 (Parallelbetrieb von Transformatoren, S. 396 f.), Kap. 9.6 (Stufenschalter, S. 421), Kap. 10.1 (Leistungsflüsse in Maschen, S. 437 f.), Kap. 11.1 (Netztopologien, S. 512–515), Kap. 11.3 (Mittelspannungsnetze, S. 520–527), S. 714.

[20] Maschinenfabrik Reinhausen, „Regelbarer Ortsnetztransformator“, Vortrag, Lehrstuhl für Elektrische Energiesysteme, FAU Erlangen-Nürnberg. Online: https://www.ees.tf.fau.de/files/2017/12/MR_Regelbarer_Ortsnetztrafo.pdf

[21] Maschinenfabrik Reinhausen, „ECOTAP VPD (24 kV)“, Produktinformation. Online: https://www.reinhausen.com/fileadmin/downloadcenter/products/oltc/ecotap_vpd/flyer/ecotap_vpd_24kv_de.pdf